# Nemotron v8 — Improved Training Pipeline
## Key Improvements over v7.5:
1. **DoRA (Weight-Decomposed LoRA)** — decomposes weight into magnitude + direction for better fine-tuning at same rank
2. **RSLoRA** — rank-stabilized scaling to get more out of the rank=32 budget (competition max)
3. **NEFTune** — adds noise to embeddings during training for better generalization
4. **Fixed `<think>` tag** — consistent opening/closing tags in CoT
5. **Fixed nested brace regex** — proper brace-balanced `\boxed{}` extraction
6. **Data quality filtering** — deduplication + CoT length filtering (100-6000 chars)
7. **Two-stage training** — Stage 1: SFT, Stage 2: GRPO (reinforcement learning)
8. **Curriculum learning** — trains on shorter/easier examples first, then harder ones
9. **Validation split** — 5% holdout for monitoring overfitting
10. **Cosine annealing with restarts** — better LR schedule for multi-epoch training

## Configuration & Mode Selection

In [ ]:
import os, sys

os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

# ============================================================
# MODE SELECTION — set exactly one to 1
# ============================================================
TRAIN_ON_KAGGLE = 1       # Mode A: Full training pipeline (SFT + optional GRPO)
USE_PRETRAINED = 0        # Mode B: Load pre-trained adapter and package

# ============================================================
# TRAINING STAGE CONTROL
# ============================================================
RUN_SFT = True            # Stage 1: Supervised Fine-Tuning
RUN_GRPO = True           # Stage 2: GRPO Reinforcement Learning (after SFT)

# ============================================================
# MODEL & PATHS
# ============================================================
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
PRETRAINED_ADAPTER_DATASET_PATH = "/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection"
DATASET_PATH = "/kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv"

assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1, "Set exactly one mode to 1."

print(f"Mode: {'TRAIN' if TRAIN_ON_KAGGLE else 'PRETRAINED'}")
print(f"Stages: SFT={RUN_SFT}, GRPO={RUN_GRPO}")

## Environment Setup (Kaggle-specific)

In [ ]:
# ============================================================
# Triton wheel installation
# ============================================================
import glob, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)

if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")

target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps",
     "--target", target, "--upgrade", "--ignore-installed", candidates[0]],
    check=True,
)

if target not in sys.path:
    sys.path.insert(0, target)
site.addsitedir(target)

import importlib.util
print("triton spec:", importlib.util.find_spec("triton"))

In [ ]:
# ============================================================
# Blackwell ptxas fix + Triton compiler patch
# ============================================================
if TRAIN_ON_KAGGLE:
    import shutil, stat

    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')

    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst

        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')

    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'
    print('Triton/ptxas environment fixes applied.')

In [ ]:
# ============================================================
# Install packages from offline wheels
# ============================================================
if TRAIN_ON_KAGGLE:
    def recursive_wheels(pattern):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")

    print("Found mamba wheels:", all_mamba)
    print("Found causal-conv1d wheels:", all_causal)

    import torch
    print(f"Python: {sys.version}")
    print(f"Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}, CUDA ver: {torch.version.cuda}")

    if not torch.cuda.is_available():
        raise RuntimeError("GPU required for training.")
    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel dir not found: {packages_dir}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "--no-index", "--find-links", packages_dir,
         "unsloth", "trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes"],
        check=True,
    )

    causal_wheel = all_causal[-1] if all_causal else None
    mamba_wheel = all_mamba[-1] if all_mamba else None

    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    else:
        raise FileNotFoundError("No compatible mamba_ssm wheel found.")

    print("Package installation complete.")

## Model Loading

In [ ]:
# ============================================================
# Load base model with Unsloth
# ============================================================
if TRAIN_ON_KAGGLE:
    import torch
    import kagglehub
    from unsloth import FastLanguageModel

    MAX_SEQ_LEN = 8192
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=False,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print("Base model loaded successfully.")

## Improved LoRA Configuration (DoRA + RSLoRA)
**Changes from v7.5:**
- **DoRA** (Weight-Decomposed LoRA): Decomposes pre-trained weights into magnitude and direction, then only fine-tunes the direction component with LoRA. Consistently outperforms standard LoRA on reasoning tasks.
- **Rank 32** (competition max) + **Alpha 64** (2x ratio): Maximizes capacity within the allowed limit
- **RSLoRA**: Rank-stabilized scaling (`alpha/sqrt(r)` instead of `alpha/r`) — prevents gradient explosion
- **All Mamba-specific modules**: `x_proj`, `dt_proj`, `out_proj` critical for the hybrid SSM architecture

> **Note:** Competition enforces `max_lora_rank = 32`. DoRA + RSLoRA squeeze more performance out of the same rank budget.

In [ ]:
# ============================================================
# Create LoRA adapter — DoRA + Rank 32 (competition max) + RSLoRA
# ============================================================
if TRAIN_ON_KAGGLE:
    from unsloth import FastLanguageModel

    # --- LoRA Hyperparameters (rank capped at competition max of 32) ---
    LORA_RANK = 32           # Competition max_lora_rank = 32
    LORA_ALPHA = 64          # 2x rank ratio
    LORA_DROPOUT = 0.05

    # Comprehensive target modules for Nemotron's hybrid Mamba+Attention architecture
    TARGET_MODULES = [
        # Attention projections
        "q_proj", "k_proj", "v_proj", "o_proj",
        # General projections (shared across layers)
        "in_proj", "out_proj",
        # MLP / MoE gates (critical for reasoning capacity)
        "gate_proj", "up_proj", "down_proj",
        # Mamba SSM-specific (critical for Nemotron's hybrid architecture)
        "x_proj", "dt_proj",
        # Output head (helps steer answer format)
        "lm_head",
    ]

    print(f"LoRA Config: rank={LORA_RANK}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
    print(f"Target modules: {TARGET_MODULES}")
    print("Using DoRA (Weight-Decomposed LoRA) + RSLoRA scaling")
    print("NOTE: rank=32 is the competition maximum")

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=TARGET_MODULES,
        bias="none",
        use_gradient_checkpointing=True,
        random_state=42,
        use_rslora=True,         # Rank-stabilized scaling: alpha/sqrt(r)
        use_dora=True,           # NEW: Weight-Decomposed LoRA — better than standard LoRA
    )
    model.print_trainable_parameters()

## Data Preparation (with fixes + quality filtering)

In [ ]:
# ============================================================
# Utility: Brace-balanced \boxed{} extraction (FIXED from v7.5)
# ============================================================
# v7.5 used: re.sub(r'\\boxed\{[^}]*\}', '', cot) which BREAKS on
# nested braces like \boxed{\frac{1}{2}}. This parser handles any depth.

def extract_boxed(text):
    """Extract content from \\boxed{...} handling nested braces correctly."""
    idx = text.find("\\boxed{")
    if idx == -1:
        return ""
    depth, start = 1, idx + 7
    for i in range(start, len(text)):
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
        if depth == 0:
            return text[start:i]
    return text[start:]


def remove_boxed(text):
    """Remove all \\boxed{...} occurrences (brace-balanced) from text."""
    result = text
    while "\\boxed{" in result:
        idx = result.find("\\boxed{")
        depth, start = 1, idx + 7
        end = len(result)
        for i in range(start, len(result)):
            if result[i] == '{':
                depth += 1
            elif result[i] == '}':
                depth -= 1
            if depth == 0:
                end = i + 1
                break
        result = result[:idx] + result[end:]
    return result


print("Utility functions defined: extract_boxed(), remove_boxed()")

In [ ]:
# ============================================================
# Build SFT dataset with quality filtering + curriculum ordering
# ============================================================
if TRAIN_ON_KAGGLE and RUN_SFT:
    os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
    os.environ["TORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

    import pandas as pd
    import random
    import hashlib
    import re
    from datasets import Dataset as HFDataset

    SEED = 42
    PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

    # Quality filtering thresholds
    MIN_COT_LENGTH = 100     # Minimum CoT chars (filter out trivial/empty reasoning)
    MAX_COT_LENGTH = 6000    # Maximum CoT chars (filter out degenerate/repetitive)

    df = pd.read_csv(DATASET_PATH)
    print(f"Raw dataset: {len(df)} rows")

    # Shuffle deterministically
    train_df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

    # -------------------------------------------------------
    # Build records with FIXED <think> tags and quality filter
    # -------------------------------------------------------
    records = []
    record_types = []
    record_cot_lengths = []  # For curriculum learning
    seen_hashes = set()      # For deduplication
    skipped = {"no_cot": 0, "too_short": 0, "too_long": 0, "duplicate": 0}

    for _, row in train_df.iterrows():
        prompt = str(row["prompt"])
        answer = str(row["answer"])
        cot = str(row["generated_cot"])

        # Skip empty/missing CoT
        if not cot or cot == "nan" or len(cot.strip()) < 5:
            skipped["no_cot"] += 1
            continue

        # Quality filter: CoT length
        cot_len = len(cot.strip())
        if cot_len < MIN_COT_LENGTH:
            skipped["too_short"] += 1
            continue
        if cot_len > MAX_COT_LENGTH:
            skipped["too_long"] += 1
            continue

        # Deduplication by prompt hash
        prompt_hash = hashlib.md5(prompt.strip().lower().encode()).hexdigest()
        if prompt_hash in seen_hashes:
            skipped["duplicate"] += 1
            continue
        seen_hashes.add(prompt_hash)

        # FIXED: Use brace-balanced removal instead of broken regex
        cot_cleaned = remove_boxed(cot).rstrip()

        user_content = prompt + PROMPT_SUFFIX

        # FIXED: Include opening <think> tag (was missing in v7.5!)
        # The chat template with enable_thinking=True expects the assistant
        # to produce <think>...</think> blocks. We ensure consistency here.
        assistant_content = f"<think>\n{cot_cleaned}\n</think>\n\\boxed{{{answer}}}"

        records.append({
            "messages": [
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": assistant_content},
            ]
        })
        record_types.append(str(row.get("type", "unknown")))
        record_cot_lengths.append(cot_len)

    print(f"\nSFT records after filtering: {len(records)}")
    print(f"Skipped: {skipped}")
    print(f"CoT length stats: min={min(record_cot_lengths)}, max={max(record_cot_lengths)}, "
          f"mean={sum(record_cot_lengths)/len(record_cot_lengths):.0f}")

    # -------------------------------------------------------
    # Curriculum learning: sort by CoT length (easy→hard)
    # Shorter CoT = simpler problems → model learns basics first
    # -------------------------------------------------------
    sorted_indices = sorted(range(len(records)), key=lambda i: record_cot_lengths[i])
    records = [records[i] for i in sorted_indices]
    record_types = [record_types[i] for i in sorted_indices]
    record_cot_lengths = [record_cot_lengths[i] for i in sorted_indices]

    print(f"\nCurriculum ordering applied: shortest CoT first → longest last")
    print(f"First 5 CoT lengths: {record_cot_lengths[:5]}")
    print(f"Last 5 CoT lengths: {record_cot_lengths[-5:]}")

    # -------------------------------------------------------
    # Train/validation split (5% holdout)
    # -------------------------------------------------------
    VAL_FRAC = 0.05
    n_val = max(1, int(len(records) * VAL_FRAC))
    # Take validation from the middle (not end, since end = hardest problems)
    rng = random.Random(SEED)
    val_indices = set(rng.sample(range(len(records)), n_val))
    train_indices = [i for i in range(len(records)) if i not in val_indices]

    train_records = [records[i] for i in train_indices]
    val_records = [records[i] for i in sorted(val_indices)]
    train_types = [record_types[i] for i in train_indices]

    train_dataset = HFDataset.from_list(train_records)
    val_dataset = HFDataset.from_list(val_records)

    print(f"\nTrain: {len(train_records)}, Validation: {len(val_records)}")
    print("Type distribution:", dict(sorted(pd.Series(train_types).value_counts().to_dict().items())))

## Stage 1: SFT Training (with NEFTune + Cosine Restarts)
**Improvements over v7.5:**
- **NEFTune** (`neftune_noise_alpha=5`): Adds uniform noise to embedding vectors during training. Published result: +2-5% on downstream tasks with zero inference cost.
- **Cosine with restarts** LR schedule: Better than plain cosine for multi-epoch training — allows the model to escape local minima between restarts.
- **Validation evaluation** every 100 steps: Catch overfitting early.
- **Warmup ratio 0.1** (was 0.05): More stable early training with higher rank LoRA.
- **Learning rate 5e-5** (was 8e-5): Slightly lower to account for higher rank + DoRA.

In [ ]:
# ============================================================
# Stage 1: SFT Training
# ============================================================
if TRAIN_ON_KAGGLE and RUN_SFT:
    import gc
    import time
    import math
    import torch
    from collections import defaultdict
    from torch.utils.data import DataLoader, Sampler
    from trl import SFTTrainer, SFTConfig

    # -------------------------------------------------------
    # Chat template formatting function
    # -------------------------------------------------------
    def formatting_prompts_func(example):
        messages = example["messages"]
        if messages and isinstance(messages[0], dict):
            conversations = [messages]
        else:
            conversations = messages
        texts = []
        for conversation in conversations:
            try:
                text = tokenizer.apply_chat_template(
                    conversation,
                    tokenize=False,
                    add_generation_prompt=False,
                    enable_thinking=True,
                )
            except TypeError:
                text = tokenizer.apply_chat_template(
                    conversation,
                    tokenize=False,
                    add_generation_prompt=False,
                )
            texts.append(text)
        return texts

    # -------------------------------------------------------
    # Stratified batching (preserved from v7.5)
    # -------------------------------------------------------
    def build_stratified_index_order(labels, batch_size, seed):
        by_label = defaultdict(list)
        for idx, label in enumerate(labels):
            by_label[label].append(idx)
        rng = random.Random(seed)
        for idx_list in by_label.values():
            rng.shuffle(idx_list)
        n_batches = max(1, math.ceil(len(labels) / batch_size))
        batches = [[] for _ in range(n_batches)]
        batch_order = list(range(n_batches))
        rng.shuffle(batch_order)
        assigned = 0
        for label in sorted(by_label.keys()):
            for idx in by_label[label]:
                batches[batch_order[assigned % n_batches]].append(idx)
                assigned += 1
        return [idx for batch in batches for idx in batch]

    class PrecomputedOrderSampler(Sampler):
        def __init__(self, order):
            self.order = list(order)
        def __iter__(self):
            return iter(self.order)
        def __len__(self):
            return len(self.order)

    class StratifiedSFTTrainer(SFTTrainer):
        def __init__(self, *args, stratified_order=None, **kwargs):
            super().__init__(*args, **kwargs)
            self.stratified_order = stratified_order
        def get_train_dataloader(self):
            if self.train_dataset is None:
                raise ValueError("Trainer requires a train_dataset.")
            if self.stratified_order is None:
                return super().get_train_dataloader()
            dataloader_kwargs = {
                "batch_size": self.args.per_device_train_batch_size,
                "sampler": PrecomputedOrderSampler(self.stratified_order),
                "collate_fn": self.data_collator,
                "num_workers": self.args.dataloader_num_workers,
                "pin_memory": self.args.dataloader_pin_memory,
                "persistent_workers": self.args.dataloader_persistent_workers,
                "drop_last": self.args.dataloader_drop_last,
            }
            if self.args.dataloader_num_workers > 0:
                dataloader_kwargs["prefetch_factor"] = self.args.dataloader_prefetch_factor
            return DataLoader(self.train_dataset, **dataloader_kwargs)

    # -------------------------------------------------------
    # Training configuration — IMPROVED
    # -------------------------------------------------------
    training_args = SFTConfig(
        output_dir="/kaggle/working/sft_output",

        # Epochs & batching
        num_train_epochs=3,                      # v7.5 used 2 — more epochs with validation monitoring
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,           # effective batch size = 8

        # Learning rate — IMPROVED
        learning_rate=5e-5,                      # v7.5 used 8e-5 — lower for DoRA + higher rank
        lr_scheduler_type="cosine_with_restarts",  # v7.5 used "cosine" — restarts help escape local minima
        warmup_ratio=0.10,                       # v7.5 used 0.05 — more warmup for stability
        lr_scheduler_kwargs={"num_cycles": 2},   # 2 restart cycles across training

        # Sequence length
        max_length=8192,

        # Optimizer — memory efficient
        optim="paged_adamw_8bit",
        adam_beta1=0.9,
        adam_beta2=0.95,
        adam_epsilon=1e-8,
        weight_decay=0.01,
        max_grad_norm=1.0,

        # NEFTune — NEW: adds noise to embeddings for better generalization
        neftune_noise_alpha=5.0,

        # Evaluation — NEW: monitor overfitting
        eval_strategy="steps",
        eval_steps=100,

        # Logging & saving
        logging_steps=10,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=2,                      # Keep only best 2 checkpoints

        # Precision & memory
        bf16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": True},
        dataloader_num_workers=2,
        remove_unused_columns=False,
        seed=SEED,
        report_to="none",
        packing=False,
        dataset_num_proc=4,

        # Load best model at end based on eval loss
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
    )

    # Build stratified order
    effective_batch_size = max(1, training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)
    stratified_order = build_stratified_index_order(train_types, effective_batch_size, SEED)
    print(f"Effective batch size: {effective_batch_size}")

    # Create trainer
    trainer = StratifiedSFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        formatting_func=formatting_prompts_func,
        stratified_order=stratified_order,
    )

    # Clear cache and train
    torch.cuda.empty_cache()
    gc.collect()

    print("=" * 60)
    print("Starting Stage 1: SFT Training")
    print(f"  DoRA: ON | NEFTune alpha: 5.0 | LR: 5e-5")
    print(f"  Rank: {LORA_RANK} | Alpha: {LORA_ALPHA} | RSLoRA: ON")
    print(f"  Epochs: 3 | Scheduler: cosine_with_restarts (2 cycles)")
    print(f"  Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")
    print("=" * 60)

    t0 = time.time()
    trainer.train()
    elapsed = time.time() - t0
    print(f"\nSFT training completed in {elapsed/60:.1f} min")

    # Save SFT adapter
    SFT_ADAPTER_DIR = "/kaggle/working/sft_adapter"
    model.save_pretrained(SFT_ADAPTER_DIR)
    tokenizer.save_pretrained(SFT_ADAPTER_DIR)
    print(f"SFT adapter saved to {SFT_ADAPTER_DIR}")

## Stage 2: GRPO (Group Relative Policy Optimization)
**Why GRPO after SFT?**
- SFT teaches the model the *format* (think + boxed answer) and general reasoning
- GRPO teaches the model to *get the right answer* on competition-specific problems
- Uses reward signal: correct `\boxed{}` match = reward 1.0, wrong = 0.0
- Additional format reward for proper `<think>...</think>` structure
- Group-based: generates N completions per prompt, ranks them by reward, updates policy

**Key settings:**
- `num_generations=4`: Generate 4 completions per prompt, learn from the best
- `max_completion_length=4096`: Enough for reasoning + answer
- `beta=0.04`: KL penalty to prevent divergence from SFT policy

In [ ]:
# ============================================================
# Stage 2: GRPO Reinforcement Learning
# ============================================================
if TRAIN_ON_KAGGLE and RUN_GRPO:
    import gc
    import time
    import re
    import torch
    import pandas as pd
    from datasets import Dataset as HFDataset

    # Reload the competition training data for GRPO
    # (GRPO uses prompts + ground truth answers, NOT the CoT)
    grpo_df = pd.read_csv(DATASET_PATH)
    print(f"GRPO dataset: {len(grpo_df)} problems")

    PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

    # Build GRPO dataset: just prompts + ground truth answers
    grpo_records = []
    for _, row in grpo_df.iterrows():
        prompt = str(row["prompt"])
        answer = str(row["answer"])
        if not answer or answer == "nan":
            continue
        grpo_records.append({
            "prompt": prompt + PROMPT_SUFFIX,
            "ground_truth": answer.strip(),
        })

    grpo_dataset = HFDataset.from_list(grpo_records)
    print(f"GRPO records: {len(grpo_records)}")

    # -------------------------------------------------------
    # Reward functions for GRPO
    # -------------------------------------------------------
    def accuracy_reward(completions, ground_truth, **kwargs):
        """
        Primary reward: Does the model's \\boxed{} answer match ground truth?
        Returns 1.0 for correct, 0.0 for incorrect.
        Uses brace-balanced extraction for robustness.
        """
        rewards = []
        for completion, gt in zip(completions, ground_truth):
            predicted = extract_boxed(completion).strip()
            expected = gt.strip()

            # Exact string match
            if predicted == expected:
                rewards.append(1.0)
                continue

            # Numeric tolerance: ±1e-2
            try:
                pred_num = float(predicted)
                exp_num = float(expected)
                if abs(pred_num - exp_num) < 1e-2:
                    rewards.append(1.0)
                    continue
            except (ValueError, TypeError):
                pass

            rewards.append(0.0)
        return rewards

    def format_reward(completions, **kwargs):
        """
        Secondary reward: Does the completion have proper format?
        +0.2 for having <think>...</think> block
        +0.1 for having \\boxed{} at the end
        Encourages structured reasoning even when answer is wrong.
        """
        rewards = []
        for completion in completions:
            score = 0.0
            # Check for think block
            if "<think>" in completion and "</think>" in completion:
                think_start = completion.find("<think>")
                think_end = completion.find("</think>")
                if think_start < think_end:
                    score += 0.2
            # Check for boxed answer
            if "\\boxed{" in completion:
                score += 0.1
            rewards.append(score)
        return rewards

    print("Reward functions defined: accuracy_reward (0/1), format_reward (0-0.3)")

    # -------------------------------------------------------
    # GRPO Training
    # -------------------------------------------------------
    from trl import GRPOTrainer, GRPOConfig

    # Put model in training mode for GRPO
    FastLanguageModel.for_training(model)

    grpo_config = GRPOConfig(
        output_dir="/kaggle/working/grpo_output",

        # GRPO-specific
        num_generations=4,            # Generate 4 completions per prompt
        max_completion_length=4096,   # Max tokens for model to generate
        max_prompt_length=2048,       # Max tokens for prompt

        # KL penalty — prevents diverging too far from SFT policy
        beta=0.04,

        # Training
        num_train_epochs=1,           # 1 epoch of RL is usually enough
        per_device_train_batch_size=1,  # Small batch due to generation overhead
        gradient_accumulation_steps=8,  # Effective batch = 8
        learning_rate=1e-5,           # Much lower LR for RL fine-tuning
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,

        # Optimizer
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        max_grad_norm=0.5,            # Tighter gradient clipping for RL stability

        # Precision & memory
        bf16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": True},

        # Logging
        logging_steps=5,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        report_to="none",
        seed=SEED,
    )

    grpo_trainer = GRPOTrainer(
        model=model,
        args=grpo_config,
        train_dataset=grpo_dataset,
        processing_class=tokenizer,
        reward_funcs=[accuracy_reward, format_reward],
    )

    # Clear cache and train
    torch.cuda.empty_cache()
    gc.collect()

    print("=" * 60)
    print("Starting Stage 2: GRPO Training")
    print(f"  Generations per prompt: 4 | Beta (KL): 0.04")
    print(f"  LR: 1e-5 | Max completion: 4096 tokens")
    print(f"  Rewards: accuracy (0/1) + format (0-0.3)")
    print(f"  Problems: {len(grpo_dataset)}")
    print("=" * 60)

    t0 = time.time()
    grpo_trainer.train()
    elapsed = time.time() - t0
    print(f"\nGRPO training completed in {elapsed/60:.1f} min")

    # Save final adapter (SFT + GRPO combined)
    GRPO_ADAPTER_DIR = "/kaggle/working/grpo_adapter"
    model.save_pretrained(GRPO_ADAPTER_DIR)
    tokenizer.save_pretrained(GRPO_ADAPTER_DIR)
    print(f"GRPO adapter saved to {GRPO_ADAPTER_DIR}")

## Mode B: Load Pre-trained LoRA

In [ ]:
# ============================================================
# Mode B: Use pre-trained adapter
# ============================================================
if USE_PRETRAINED:
    SRC_ADAPTER_DIR = PRETRAINED_ADAPTER_DATASET_PATH
    required_files = ["adapter_config.json", "adapter_model.safetensors"]

    print("Using pre-trained adapter from:", SRC_ADAPTER_DIR)
    for fname in required_files:
        fpath = os.path.join(SRC_ADAPTER_DIR, fname)
        if not os.path.exists(fpath):
            raise FileNotFoundError(f"Missing required adapter file: {fpath}")
        print(f"  {fname}: {os.path.getsize(fpath)/1024/1024:.1f} MB")

## Package submission.zip

In [ ]:
# ============================================================
# Create submission.zip
# ============================================================
import json, shutil, zipfile

OUTPUT_DIR = "/kaggle/working"
SUBMISSION_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "submission_adapter")
os.makedirs(SUBMISSION_ADAPTER_DIR, exist_ok=True)

required_files = ["adapter_config.json", "adapter_model.safetensors"]

# Determine source adapter directory
if TRAIN_ON_KAGGLE:
    if RUN_GRPO:
        src_adapter_dir = "/kaggle/working/grpo_adapter"
        print("Packaging GRPO-trained adapter (SFT + RL)")
    else:
        src_adapter_dir = "/kaggle/working/sft_adapter"
        print("Packaging SFT-trained adapter")
else:
    src_adapter_dir = PRETRAINED_ADAPTER_DATASET_PATH
    print("Packaging pre-trained adapter from:", src_adapter_dir)

# Copy adapter files
for fname in required_files:
    src = os.path.join(src_adapter_dir, fname)
    dst = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing: {src}")
    shutil.copy2(src, dst)
    print(f"Copied {fname} ({os.path.getsize(dst)/1024/1024:.1f} MB)")

# Patch config for inference
config_path = os.path.join(SUBMISSION_ADAPTER_DIR, "adapter_config.json")
with open(config_path, "r") as f:
    cfg = json.load(f)

cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"] = 0.0  # Disable dropout at inference

with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

# Create zip
zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        fpath = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
        zf.write(fpath, fname)
        print(f"  Added {fname}")

zip_sz = os.path.getsize(zip_path) / 1024 / 1024
print(f"\nsubmission.zip: {zip_sz:.1f} MB")
print("Done! Ready to submit.")